# 00 · Setup & Sample Data

Run this notebook first. It connects to your Atlas cluster and loads a small
set of **synthetic** AMEX-style banking data (customers + card transactions)
that Modules 1 and 3 build on. Nothing here touches real AMEX data.

**This repo covers four demo modules**, matching the AMEX operational-efficiency
demo plan:

| # | Notebook | Story |
|---|----------|-------|
| 1 | `01_module1_ha_failover.ipynb` | Zero-data-loss failover under live write load |
| 2 | `02_module2_millisecond_analytics.ipynb` | Millisecond aggregation queries + leadership dashboard |
| 3 | `03_module3_governed_indexing.ipynb` | Performance Advisor, query blocking, schema validation |
| 4 | `04_module4_eventing_vector_search.ipynb` | Change Streams + Vector Search for AI readiness |

Plus two bonus notebooks: Queryable Encryption and multi-region/sharding.

**Runs locally or in Google Colab** — the first code cell in every notebook
detects which environment it's in and adapts (installs packages, prompts for
your connection string instead of reading `.env`).

Running locally? Make sure the cluster is **M10+** — Test Failover, Query
Insights, and Operation Rejection Filters aren't available on Free/Flex tier.


In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pymongo[encryption]", "certifi", "python-dotenv", "requests", "matplotlib", "pandas", "faker"],
        check=True,
    )
    from getpass import getpass
    ATLAS_URI = os.environ.get("ATLAS_URI") or getpass("Atlas connection string (ATLAS_URI): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    ATLAS_URI = os.environ["ATLAS_URI"]

DEMO_DB = os.environ.get("DEMO_DB", "amex_demo")
import certifi
CA_FILE = certifi.where()
print("Environment:", "Colab" if IN_COLAB else "local", "| DB:", DEMO_DB)


In [ ]:
from pymongo import MongoClient

client = MongoClient(ATLAS_URI, tlsCAFile=CA_FILE)
print("Connected. Databases visible to this user:", client.list_database_names())

db = client[DEMO_DB]


## Generate synthetic customers and transactions

~2,000 customers and ~50,000 card transactions. The `customers` collection
includes an `ssn` and `account_number` field — these are the fields the
Queryable Encryption bonus notebook protects.


In [ ]:
import random
from datetime import datetime, timedelta, timezone
from faker import Faker

fake = Faker()
Faker.seed(42)
random.seed(42)

N_CUSTOMERS = 2000
N_TRANSACTIONS = 50_000
MERCHANTS = [
    "Amazon", "Whole Foods", "Shell Gas", "Delta Air Lines", "Starbucks",
    "Best Buy", "Uber", "Netflix", "Target", "Costco", "Apple Store",
    "Marriott", "Chipotle", "Home Depot", "CVS Pharmacy",
]

def make_customer(i):
    return {
        "customer_id": f"CUST{i:06d}",
        "name": fake.name(),
        "email": fake.email(),
        "ssn": fake.ssn(),
        "account_number": fake.iban(),
        "city": fake.city(),
        "state": fake.state_abbr(),
        "joined": fake.date_time_between(start_date="-8y", end_date="-1y"),
    }

customers = [make_customer(i) for i in range(N_CUSTOMERS)]
print("Sample customer:", customers[0])


In [ ]:
def make_transaction(i, customer):
    ts = datetime.now(timezone.utc) - timedelta(days=random.randint(0, 365), minutes=random.randint(0, 1440))
    return {
        "transaction_id": f"TXN{i:08d}",
        "customer_id": customer["customer_id"],
        "account_id": customer["account_number"],
        "amount": round(random.uniform(3, 1500), 2),
        "merchant": random.choice(MERCHANTS),
        "status": random.choices(["approved", "declined", "pending"], weights=[92, 5, 3])[0],
        "timestamp": ts,
    }

transactions = [
    make_transaction(i, random.choice(customers))
    for i in range(N_TRANSACTIONS)
]
print("Sample transaction:", transactions[0])


In [ ]:
# Reset collections so this notebook is safe to re-run before the session.
db.customers.drop()
db.transactions.drop()

db.customers.insert_many(customers)

BATCH = 5000
for start in range(0, len(transactions), BATCH):
    db.transactions.insert_many(transactions[start:start + BATCH])

print("customers:", db.customers.count_documents({}))
print("transactions:", db.transactions.count_documents({}))


Data's loaded. Next: open **01_module1_ha_failover.ipynb**.
